In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

IN_PATH = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv")
OUT_DIR = Path("/Users/joshmacbook/python_projects/OAD/Data/MST")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_17_23 = OUT_DIR / "combined_eventlevel_pga_2017_2023_mean.csv"
OUT_24P   = OUT_DIR / "combined_roundlevel_2024_present.csv"

df = pd.read_csv(IN_PATH)

# normalize
df["tour"] = df["tour"].astype(str).str.lower().str.strip()
df["year"] = pd.to_numeric(df["year"], errors="coerce")

# -----------------------------
# 2024-present: leave as-is
# -----------------------------
df_24p = df[df["year"] >= 2024].copy()
df_24p.to_csv(OUT_24P, index=False)

# -----------------------------
# 2017-2023: PGA only, event-level
# -----------------------------
df_17_23 = df[(df["year"] >= 2017) & (df["year"] <= 2023) & (df["tour"] == "pga")].copy()

# columns to drop
drop_cols = [
    "round_num",
    "start_hole",
    "teetime",
    "prox_rgh",
    "prox_fw",
    "great_shots",
    "poor_shots",
    "round_date",
]
df_17_23 = df_17_23.drop(columns=[c for c in drop_cols if c in df_17_23.columns], errors="ignore")

# group keys (define one player start in one event-year)
# NOTE: intentionally NOT grouping by fin_text/finish_num to prevent accidental splits.
group_cols = [
    "tour",
    "year",
    "season",
    "event_completed",
    "event_name",
    "event_id",
    "player_name",
    "dg_id",
    "course_name",
    "course_num",
    "course_par",
]

missing = [c for c in group_cols if c not in df_17_23.columns]
if missing:
    raise ValueError(f"Missing required columns in source CSV: {missing}")

# numeric cols we want to mean (mean-everything approach)
mean_cols = [
    "round_score",
    "sg_putt", "sg_arg", "sg_app", "sg_ott", "sg_t2g", "sg_total",
    "driving_dist", "driving_acc", "gir", "scrambling",
    "eagles_or_better", "birdies", "pars", "bogies", "doubles_or_worse",
]

for c in mean_cols:
    if c in df_17_23.columns:
        df_17_23[c] = pd.to_numeric(df_17_23[c], errors="coerce")

# helpers to carry fin_text/finish_num through safely
def last_nonnull(s: pd.Series):
    s2 = s.dropna()
    return s2.iloc[-1] if len(s2) else np.nan

agg = {
    "rounds_played_event": ("dg_id", "size"),  # count of rows/rounds in the group
}

# mean for each numeric stat
for c in mean_cols:
    if c in df_17_23.columns:
        agg[c] = (c, "mean")

# carry forward finish fields (no aggregation requested, but we avoid grouping by them)
if "fin_text" in df_17_23.columns:
    agg["fin_text"] = ("fin_text", last_nonnull)
if "finish_num" in df_17_23.columns:
    agg["finish_num"] = ("finish_num", last_nonnull)

out_17_23 = (
    df_17_23
      .groupby(group_cols, as_index=False)
      .agg(**agg)
)

# optional: enforce int on rounds_played_event
out_17_23["rounds_played_event"] = pd.to_numeric(out_17_23["rounds_played_event"], errors="coerce").fillna(0).astype(int)

out_17_23.to_csv(OUT_17_23, index=False)

print("Wrote:")
print(" ", OUT_17_23, "rows:", len(out_17_23))
print(" ", OUT_24P,   "rows:", len(df_24p))


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_32361/140288325.py:12: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(IN_PATH)


Wrote:
  /Users/joshmacbook/python_projects/OAD/Data/MST/combined_eventlevel_pga_2017_2023_mean.csv rows: 46675
  /Users/joshmacbook/python_projects/OAD/Data/MST/combined_roundlevel_2024_present.csv rows: 77731
